# Competing hypotheses about what a delayed recounting recounts

This notebook builds the schematic figure contrasting the two hypotheses
we test:

- **Hypothesis 1**: every recounting is a noisy version of *the episode*.
  Noise may grow over time, but each recounting is generated from the
  same underlying representation.
- **Hypothesis 2**: every recounting is a noisy version of *the previous
  recounting*, so a delayed recounting is generated from the immediate
  one rather than from the episode.

Panels A and C sample from each generative model in a two-dimensional
cartoon of the semantic space. Panels B and D show the three distance
distributions those samples imply, computed from exactly the points
drawn in panels A and C.

Nothing here is fit to data: the figure illustrates the predictions the
two hypotheses make, which the analyses in the rest of the paper
adjudicate between.

## Imports

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from scipy.stats import gaussian_kde

from analysis_helpers.constants import FIG_DIR

## Parameters

Each recounting is drawn from an isotropic Gaussian. The episode's own
spread (the gray halo in panels A and C) stands in for the fact that
even the stimulus is sampled imperfectly by any one viewing; the
immediate recounting is drawn with a wider spread, and the delayed
recounting wider still. Only the *center* of the delayed distribution
differs between the two hypotheses.

`SELECTED_IMM_RADIUS` sets how far from the episode the participant's
actual immediate recounting falls. It's the one parameter the figure is
sensitive to: the closer that point sits to the episode, the harder it
is to tell the two hypotheses apart, since the two candidate centers
converge.

In [ ]:
RANDOM_SEED = 0
N_SAMPLES = 150

EPISODE_SIGMA = 0.55
IMMEDIATE_SIGMA = 1.00
DELAYED_SIGMA = 1.50
SELECTED_IMM_RADIUS = 2.20

EPISODE_COLOR = '#3a3a3a'
IMMEDIATE_COLOR = '#c44e52'
DELAYED_COLOR = '#4c72b0'
BETWEEN_COLOR = '#8a8f98'

CLOUD_MAX_ALPHA = 0.46
EPISODE_CLOUD_ALPHA = 0.78
CLOUD_GRID_N = 400
CLOUD_EXTENT = 9.0
AXES_LIM = 6.05

DOT_SIZE = 11
DOT_ALPHA = 0.45
SELECTED_DOT_SIZE = 80
SELECTED_EDGECOLOR = '#111'
SELECTED_LINEWIDTH = 1.1
EPISODE_DOT_SIZE = 105

KDE_FILL_ALPHA = 0.35
KDE_LINEWIDTH = 1.8
KDE_GRID_N = 512
KDE_XMAX = 7.0

PANEL_LABEL_FONTSIZE = 17
ROW_LABEL_FONTSIZE = 14
CALLOUT_FONTSIZE = 11.5
LEGEND_FONTSIZE = 11
AXIS_LABEL_FONTSIZE = 12.5
TICKLABEL_FONTSIZE = 11

CALLOUT_COLOR = '#555'
CALLOUT_LINEWIDTH = 0.9
KDE_BW_METHOD = 0.5

# callouts sit in the band between the sketch panels, each centered on
# the end of its own leader line: (label, x as a fraction of the panel's
# width)
CALLOUT_BAND_Y = 0.5
CALLOUTS = (('Original\nepisode', 0.06),
            ('Immediate\nrecounting', 0.53),
            ('Delayed\nrecounting', 0.97))

ROW_LABELS = (
    'Hypothesis 1: recountings are\nnoisy versions of the $\\it{original}$',
    'Hypothesis 2: recountings are noisy\n'
    'versions of $\\it{previous\\ recountings}$'
)
ROW_LABEL_X = 0.074

FIGSIZE = (10, 7.8)

plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'sans:italic'
plt.rcParams['mathtext.rm'] = 'sans'

## Simulate recountings under each hypothesis

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

episode = np.zeros(2)

# candidate immediate recountings: noisy versions of the episode
immediate = rng.normal(size=(N_SAMPLES, 2)) * IMMEDIATE_SIGMA

# the participant's actual immediate recounting, placed up and to the
# right so the delayed cloud it anchors under hypothesis 2 stays on-panel
imm_radii = np.linalg.norm(immediate, axis=1)
imm_angles = np.arctan2(immediate[:, 1], immediate[:, 0])
candidates = np.flatnonzero((imm_angles > np.deg2rad(20))
                            & (imm_angles < np.deg2rad(55)))
selected_imm_ix = candidates[
    np.argmin(np.abs(imm_radii[candidates] - SELECTED_IMM_RADIUS))
]
selected_imm = immediate[selected_imm_ix]

# candidate delayed recountings. The same draws are reused under both
# hypotheses, so the two clouds differ only in where they're centered.
delayed_noise = rng.normal(size=(N_SAMPLES, 2)) * DELAYED_SIGMA
delayed = {'h1': episode + delayed_noise, 'h2': selected_imm + delayed_noise}

del_radii = np.linalg.norm(delayed_noise, axis=1)
del_angles = np.arctan2(delayed_noise[:, 1], delayed_noise[:, 0])
candidates = np.flatnonzero((del_angles > np.deg2rad(-70))
                            & (del_angles < np.deg2rad(-25)))
selected_del_ix = candidates[
    np.argmin(np.abs(del_radii[candidates] - DELAYED_SIGMA))
]

dists = {
    hyp: {
        'imm_ep': np.linalg.norm(immediate - episode, axis=1),
        'del_ep': np.linalg.norm(delayed[hyp] - episode, axis=1),
        'del_imm': np.linalg.norm(delayed[hyp] - selected_imm, axis=1)
    }
    for hyp in ('h1', 'h2')
}

for hyp, hyp_dists in dists.items():
    print(f'{hyp}: ' + ', '.join(f'mean {k} distance: {v.mean():.3f}'
                                 for k, v in hyp_dists.items()))

## Functions

In [ ]:
def draw_cloud(ax, center, sigma, color, max_alpha=CLOUD_MAX_ALPHA):
    """Render a sampling distribution as a Gaussian fading out from `center`."""
    grid = np.linspace(-CLOUD_EXTENT, CLOUD_EXTENT, CLOUD_GRID_N)
    xx, yy = np.meshgrid(grid, grid)
    sq_dists = (xx - center[0]) ** 2 + (yy - center[1]) ** 2

    rgba = np.zeros((CLOUD_GRID_N, CLOUD_GRID_N, 4))
    rgba[..., :3] = mcolors.to_rgb(color)
    rgba[..., 3] = max_alpha * np.exp(-sq_dists / (2 * sigma ** 2))
    ax.imshow(rgba,
              extent=(-CLOUD_EXTENT, CLOUD_EXTENT,
                      -CLOUD_EXTENT, CLOUD_EXTENT),
              origin='lower',
              interpolation='bilinear',
              zorder=0)


def draw_sketch(ax, hyp):
    """Draw the sampling cartoon for one hypothesis (panel A or panel C)."""
    delayed_center = episode if hyp == 'h1' else selected_imm

    draw_cloud(ax, delayed_center, DELAYED_SIGMA, DELAYED_COLOR)
    draw_cloud(ax, episode, IMMEDIATE_SIGMA, IMMEDIATE_COLOR)
    draw_cloud(ax, episode, EPISODE_SIGMA, EPISODE_COLOR,
               max_alpha=EPISODE_CLOUD_ALPHA)

    # candidate recountings
    ax.scatter(*delayed[hyp].T, s=DOT_SIZE, color=DELAYED_COLOR,
               alpha=DOT_ALPHA, linewidths=0, zorder=2)
    ax.scatter(*immediate.T, s=DOT_SIZE, color=IMMEDIATE_COLOR,
               alpha=DOT_ALPHA, linewidths=0, zorder=3)

    # the participant's actual recountings, and the episode itself
    ax.scatter(*delayed[hyp][selected_del_ix], s=SELECTED_DOT_SIZE,
               color=DELAYED_COLOR, edgecolors=SELECTED_EDGECOLOR,
               linewidths=SELECTED_LINEWIDTH, zorder=4)
    ax.scatter(*selected_imm, s=SELECTED_DOT_SIZE, color=IMMEDIATE_COLOR,
               edgecolors=SELECTED_EDGECOLOR, linewidths=SELECTED_LINEWIDTH,
               zorder=5)
    ax.scatter(*episode, s=EPISODE_DOT_SIZE, color=EPISODE_COLOR, zorder=6)

    ax.set_xlim(-AXES_LIM, AXES_LIM)
    ax.set_ylim(-AXES_LIM, AXES_LIM)
    ax.set_aspect('equal')
    ax.axis('off')


def draw_kdes(ax, hyp):
    """Draw the three distance distributions implied by one hypothesis."""
    grid = np.linspace(0, KDE_XMAX, KDE_GRID_N)
    for key, color, label in (
            ('imm_ep', IMMEDIATE_COLOR, 'Immediate vs. episode'),
            ('del_ep', DELAYED_COLOR, 'Delayed vs. episode'),
            ('del_imm', BETWEEN_COLOR, 'Delayed vs. immediate')
    ):
        density = gaussian_kde(dists[hyp][key], bw_method=KDE_BW_METHOD)(grid)
        ax.fill_between(grid, density, color=color, alpha=KDE_FILL_ALPHA, lw=0)
        ax.plot(grid, density, color=color, lw=KDE_LINEWIDTH, label=label)

    ax.set_xlim(0, KDE_XMAX)
    ax.set_ylim(bottom=0)
    ax.set_ylabel('Density', fontsize=AXIS_LABEL_FONTSIZE)
    ax.tick_params(labelsize=TICKLABEL_FONTSIZE)
    sns.despine(ax=ax)


def add_callout(text, xy_h1, xy_h2, text_xy):
    """Label a point in both sketch panels from one shared text block."""
    for ax, xy in ((a_ax, xy_h1), (c_ax, xy_h2)):
        ax.annotate('',
                    xy=xy,
                    xycoords=ax.transData,
                    xytext=text_xy,
                    textcoords=fig.transFigure,
                    arrowprops={'arrowstyle': '-',
                                'color': CALLOUT_COLOR,
                                'lw': CALLOUT_LINEWIDTH,
                                'shrinkA': 0,
                                'shrinkB': 4})
    fig.text(*text_xy, text,
             fontsize=CALLOUT_FONTSIZE,
             color=CALLOUT_COLOR,
             ha='center',
             va='center',
             ma='center',
             linespacing=1.4,
             bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 3})

## Construct figure

In [ ]:
################################ LAYOUT ################################

fig = plt.figure(figsize=FIGSIZE)
gs = fig.add_gridspec(3, 2,
                      width_ratios=[0.79, 1.21],
                      height_ratios=[1, 0.19, 1],
                      wspace=0.18,
                      hspace=0.0,
                      left=0.115,
                      right=0.985,
                      top=0.965,
                      bottom=0.085)
a_ax = fig.add_subplot(gs[0, 0])
b_ax = fig.add_subplot(gs[0, 1])
c_ax = fig.add_subplot(gs[2, 0])
d_ax = fig.add_subplot(gs[2, 1], sharex=b_ax, sharey=b_ax)

################################ PANELS ################################

draw_sketch(a_ax, 'h1')
draw_sketch(c_ax, 'h2')
draw_kdes(b_ax, 'h1')
draw_kdes(d_ax, 'h2')

# panels B and D share an x-axis; only the lower one is labeled
b_ax.tick_params(labelbottom=False)
d_ax.set_xlabel('Distance', fontsize=AXIS_LABEL_FONTSIZE)

b_ax.legend(fontsize=LEGEND_FONTSIZE, frameon=False, loc='upper right')

############################### CALLOUTS ###############################

# the sketch panels' equal aspect ratio shrinks their axes to squares
# narrower than the cells they sit in, so let the layout settle before
# reading off positions to place the callouts and panel labels against
fig.canvas.draw()

a_pos, c_pos = a_ax.get_position(), c_ax.get_position()
label_y = c_pos.y1 + CALLOUT_BAND_Y * (a_pos.y0 - c_pos.y1)

callout_points = (
    (episode, episode),
    (selected_imm, selected_imm),
    (delayed['h1'][selected_del_ix], delayed['h2'][selected_del_ix])
)
for (label, x_frac), (xy_h1, xy_h2) in zip(CALLOUTS, callout_points):
    add_callout(label, xy_h1, xy_h2,
                (a_pos.x0 + x_frac * a_pos.width, label_y))

############################## ROW LABELS ##############################

for row_label, ax in zip(ROW_LABELS, (a_ax, c_ax)):
    pos = ax.get_position()
    fig.text(ROW_LABEL_X,
             (pos.y0 + pos.y1) / 2,
             row_label,
             fontsize=ROW_LABEL_FONTSIZE,
             rotation=90,
             ha='center',
             va='center',
             ma='center',
             linespacing=1.5)

############################# PANEL LABELS #############################

for label, ax, dx in (('A', a_ax, 0.015), ('B', b_ax, 0.055),
                      ('C', c_ax, 0.015), ('D', d_ax, 0.055)):
    fig.text(ax.get_position().x0 - dx,
             ax.get_position().y1 - 0.028,
             label,
             fontsize=PANEL_LABEL_FONTSIZE,
             fontweight='semibold',
             ha='right',
             va='bottom')

plt.savefig(FIG_DIR / 'hypotheses.pdf', bbox_inches='tight')

plt.show()